# Benchmarking QPUs from CUDA-Q via qBraid

With qBraid now wired into CUDA-Q, a single `@cudaq.kernel` can be dispatched to any device qBraid brokers — Rigetti, IonQ, IQM, and more — by changing one string. That makes it natural to use CUDA-Q as the *driver* for cross-vendor QPU benchmarking: write the kernel once, run it against the ideal qBraid QIR simulator to fix a baseline, then run the same compiled OpenQASM 2 program against real hardware and score the gap.

This notebook walks through the workflow on a textbook benchmarking circuit: the **GHZ state**. The ideal distribution of $|0\rangle^{\otimes n}+|1\rangle^{\otimes n}$ is exactly two bitstrings with equal probability, so a simple, intuitive score — *GHZ fidelity* — drops off the moment noise creeps in.

Outline:
1. The benchmark kernel + a scoring function
2. Ideal baseline on the qBraid QIR simulator
3. Same kernel against another qBraid-brokered device
4. Async submission and result persistence for long QPU queues
5. Reporting & extension ideas

## Setup

Same as the quickstart: a CUDA-Q build with the qBraid target and a qBraid API key. Browse devices at <https://account.qbraid.com/devices> and paste any device IDs you want to benchmark into the `QPU_DEVICE_IDS` list below. Leave it empty and the notebook still runs end-to-end against simulators only.

In [ ]:
# SKIP_CI: requires CUDA-Q runtime and qBraid API credentials
import os
from dataclasses import dataclass
from time import perf_counter

import cudaq

assert os.environ.get("QBRAID_API_KEY"), "Export QBRAID_API_KEY first."

# qBraid QIR state-vector simulator — the default target. Ideal, free of
# noise, unlimited submissions, so it makes a clean reference.
QBRAID_SIM = "qbraid:qbraid:sim:qir-sv"

# A second simulator from a different vendor — same circuit, different
# backend, still ideal. Useful as a sanity check that vendor routing works.
AWS_SIM = "aws:aws:sim:sv1"

# Paste QPU device IDs here (e.g. IonQ Aria, Rigetti Ankaa, IQM Garnet via
# qBraid) to include real hardware in the sweep. Every ID in this list is
# benchmarked across every width in WIDTHS. Leave the list empty to skip
# the hardware sweep. Look up current IDs at https://account.qbraid.com/devices.
QPU_DEVICE_IDS: list[str] = [
    "aws:iqm:qpu:garnet",
    "aws:rigetti:qpu:cepheus-1-108q",
    # "aws:rigetti:qpu:ankaa-9q-3",
]

SHOTS = 100

## 1. The benchmark kernel + scoring

A width-`n` GHZ state. CUDA-Q kernels take parameters, so we can sweep `n` without rewriting the circuit.

In [ ]:
@cudaq.kernel
def ghz(n: int):
    q = cudaq.qvector(n)
    h(q[0])
    for i in range(n - 1):
        x.ctrl(q[i], q[i + 1])
    mz(q)


def ghz_fidelity(counts, n: int) -> float:
    """Probability mass that landed on the two ideal bitstrings.

    For a perfect GHZ this is 1.0; uniform random noise on n=4 would give ~0.125.
    """
    total = sum(counts.values())
    if total == 0:
        return float("nan")
    all_zero = "0" * n
    all_one = "1" * n
    return (counts.get(all_zero, 0) + counts.get(all_one, 0)) / total

## 2. The benchmarking primitive

Everything below funnels through one function: given a device ID and a width, submit the kernel via the qBraid target and return a row of results. Same code path for simulators and QPUs — the only difference is `machine`.

In [ ]:
@dataclass
class BenchRow:
    device: str
    n_qubits: int
    fidelity: float
    wall_seconds: float
    raw_counts: dict


def benchmark(device_id: str, n_qubits: int, shots: int = SHOTS) -> BenchRow:
    cudaq.set_target("qbraid", machine=device_id)
    t0 = perf_counter()
    counts = cudaq.sample(ghz, n_qubits, shots_count=shots)
    wall = perf_counter() - t0
    as_dict = {bits: c for bits, c in counts.items()}
    return BenchRow(
        device=device_id,
        n_qubits=n_qubits,
        fidelity=ghz_fidelity(as_dict, n_qubits),
        wall_seconds=wall,
        raw_counts=as_dict,
    )

## 3. Ideal baseline

Run widths 2–5 on the qBraid QIR simulator. We expect `fidelity == 1.0` on every row.

In [ ]:
WIDTHS = [2, 3, 4, 5]
rows: list[BenchRow] = []

In [ ]:
for n in WIDTHS:
    row = benchmark(QBRAID_SIM, n)
    print(f"[qBraid QIR sim, n={n}] fidelity={row.fidelity:.3f}  ({row.wall_seconds:.1f}s)")
    rows.append(row)

Cross-check against a *different* ideal simulator routed through the same qBraid target. If the two simulators disagree at this stage, the issue is upstream of the noise model — most likely circuit translation.

In [ ]:
for n in WIDTHS:
    row = benchmark(AWS_SIM, n)
    print(f"[AWS SV1 via qBraid, n={n}] fidelity={row.fidelity:.3f}  ({row.wall_seconds:.1f}s)")
    rows.append(row)

## 4. Noisy QPU run

Now the part the announcement is really about — the same `ghz` kernel, the same `benchmark()` function, but `machine=` is a real QPU. We iterate over every device ID in `QPU_DEVICE_IDS`, so adding a vendor to the comparison is a one-line edit at the top of the notebook. The drop in `fidelity` is the headline noise figure for each device on this circuit family.

In [ ]:
if not QPU_DEVICE_IDS:
    print("QPU_DEVICE_IDS is empty — skipping the hardware sweep.")
    print("Add device IDs from https://account.qbraid.com/devices to")
    print("QPU_DEVICE_IDS at the top of the notebook to include them.")
else:
    for qpu in QPU_DEVICE_IDS:
        for n in WIDTHS:
            row = benchmark(qpu, n)
            print(f"[{qpu}, n={n}] fidelity={row.fidelity:.3f}  ({row.wall_seconds:.1f}s)")
            rows.append(row)

### Tip: don't block on the QPU queue

QPU jobs on shared hardware can sit in a queue for a long time. `sample_async` returns a future immediately, and the future serializes to a string that can be written to disk and rehydrated from another process — your benchmarking driver doesn't have to keep a long-lived Python process alive.

In [ ]:
import re

for qpu in QPU_DEVICE_IDS:
    cudaq.set_target("qbraid", machine=qpu)
    future = cudaq.sample_async(ghz, 4, shots_count=SHOTS)
    slug = re.sub(r"[^A-Za-z0-9._-]+", "_", qpu)
    path = f"qpu_job_n4__{slug}.json"
    with open(path, "w") as f:
        f.write(str(future))
    print(f"Wrote {path} — pick this one up later with:")
    print(f"  cudaq.AsyncSampleResult(open('{path}').read()).get()")

## 5. Report

Fold everything we collected into a table that's easy to analyse, and a quick plot of fidelity vs. width per device.

In [ ]:
print(f"{'device':<40} {'n':>3} {'fidelity':>10} {'wall(s)':>8}")
print("-" * 65)
for r in rows:
    print(f"{r.device:<40} {r.n_qubits:>3} {r.fidelity:>10.3f} {r.wall_seconds:>8.2f}")

In [ ]:
import matplotlib.pyplot as plt

by_device: dict[str, list[BenchRow]] = {}
for r in rows:
    by_device.setdefault(r.device, []).append(r)

fig, ax = plt.subplots(figsize=(7, 4))
for device, drows in by_device.items():
    drows.sort(key=lambda r: r.n_qubits)
    ax.plot([r.n_qubits for r in drows], [r.fidelity for r in drows], marker="o", label=device)

ax.set_xlabel("GHZ width (qubits)")
ax.set_ylabel("GHZ fidelity")
ax.set_ylim(0, 1.1)
ax.legend(fontsize=8, loc="lower left")
ax.set_title("CUDA-Q × qBraid: same kernel, multiple devices")
plt.grid()
plt.tight_layout()
plt.show()

## Where to take this next

- **Other circuit families.** Swap `ghz` for QAOA on a small graph, a quantum volume circuit, or whatever family is closest to your real workload. The benchmarking primitive doesn't need to change.
- **Other scoring functions.** GHZ fidelity is intuitive but coarse. Hellinger fidelity against the ideal distribution, total variation distance, or per-qubit readout error are all swap-ins for `ghz_fidelity`.
- **`observe` instead of `sample`.** For variational workloads, `cudaq.observe(..., shots_count=...)` runs through the qBraid target the same way and returns expectation values directly.
- **Async sweeps.** Submit every (device, width) pair with `sample_async` first, persist the handles, then drain the results — pipelines QPU queues against each other instead of running serially.
- **Adding your device.** If you want your QPU exposed to every CUDA-Q user via this route, reach out to qBraid for direct integration — the device ID then drops into `machine=` and any user of this notebook can benchmark against it.

<div class="alert alert-block alert-info">
<b>Copyright Notice:</b> 
    All rights reserved © [2026] qBraid. This notebook is part of the qBraid-Lab-Demo repository.
The qBraid-Lab-Demo is licensed under the Apache License, Version 2.0.
You may obtain a copy of the License at <https://www.apache.org/licenses/LICENSE-2.0>.
Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
</div>